## 블루윙스 스마트 팩토리 -- 라인 별 3-클래스 베이스라인 모델

[기본 설계]
- 전처리는 이미 완료된 제품별 파일(A_31, TO_31) 사용, 추가 전처리 없음.
- 전처리 = 제품코드 단위 | 모델링 = 라인 단위 수행 (csv 파일 내 LINE_ 원핫 컬럼 -> 실제 라인 복원 후 분리)
- 후보 모델 3종 (XGB, CAT, LGBM) 모두 사용 모델 테이블로 비교
- 데이터 누수 방지 + 과적합 방어 적용

[타겟 설계]
- Y_Quality를 회귀로 예측 -> 학습 fold의 임계값으로 3-class 분류
- 임계값은 매 fold의 Train 데이터에서만 계산 (등급간 경계 값)
  - 테스트 폴드 정보가 임계값에 섞이지 않게 하여 누수 원천 차단

[데이터 누수 방지]
1) Y_quality는 타겟으로만 사용, 피쳐에서 제외
2) Product_id, product_code, timestamp, line_원핫 피처 제외
3) MI 기반 피처 선택은 각 fold의 train 구간에서만 계산 -> test에 적용
4) 회귀->분류 임계값도 각 fold의 train에서만 계산 (타겟 설계 참고)
5) 데이터 합성 없이 표본 가중치만 사용 -> 합성 leakage 차단

[과적합 방어]
- fold 내부 MI 피처 선택으로 p/n을 표본의 1/5 수준으로 축소
- 얕은 트리 + 강한 정규화 + 서브샘플링 + 조기종료
- train/test Macro-F1 gap을 라인x모델마다 계산해 과적합 여부 명시
- 라인별 최소 클래스 수에 따라 fold 수를 적응적으로 축소

### 라이브러리 불러오기

In [1]:
# 기본 라이브러리
import pandas as pd
import numpy as np

# 시각화
import seaborn as sns
import matplotlib.pyplot as plt
import koreanize_matplotlib

# 피쳐 선별
from sklearn.feature_selection import mutual_info_regression

# 모델링
import lightgbm as lgb
import xgboost as xgb
import catboost as cat

# 모델 평가
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, precision_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, auc

# 경고 무시
import warnings
warnings.filterwarnings('ignore')

# 시드 설정
np.random.seed(42)

### 데이터 불러오기

In [2]:
# 라인 복원
def reconstruct_line(df, dummy_cols, baseline_name):
    line = pd.Series(baseline_name, index=df.index)
    for c in dummy_cols:
        line[df[c]==1] = c.replace('LINE_','')
    return line

LINE_MAP = {
    'A_31_preprocessed.csv': (['LINE_T010306', 'LINE_T050304', 'LINE_T050307'], 'T010305'),
    'TO_31_preprocessed.csv': (['LINE_T100306'], 'T100304'),
}

# 모델링에서 drop할 피처들
DROP_BASE = ['PRODUCT_ID','Y_Class','Y_Quality','PRODUCT_CODE','TIMESTAMP']

# 데이터 불러오기
def load_data():
    frames = []
    for path, (dummies, baseline) in LINE_MAP.items():
        df = pd.read_csv(path)
        df['LINE'] = reconstruct_line(df, dummies, baseline)
        drop_cols = DROP_BASE + dummies + ['LINE']
        feats = [c for c in df.columns if c not in drop_cols]
        for ln in df['LINE'].unique():
            sub = df[df['LINE']==ln].reset_index(drop=True)
            frames.append((ln, sub, feats))
    return frames

### 후보 모델 3종</br>

- Y_Quality 예측 후 고정 임계값으로 등급 분류
- 과적합 방어 전략 : 얕은 트리 + 강한 정규화 + 서브샘플링

In [3]:
def build_model():
    return {
        'LightGBM': lgb.LGBMRegressor(
            n_estimators=800, learning_rate=0.05, num_leaves=11, max_depth=3,
            min_child_samples=10, reg_alpha=0.3, reg_lambda=0.5,
            subsample=0.7, subsample_freq=1, colsample_bytree=0.5,
            random_state=42, verbose=-1, n_jobs=-1),
        'XGBoost': xgb.XGBRegressor(
            n_estimators=800, learning_rate=0.05, max_depth=3,
            min_child_weight=5, reg_alpha=0.3, reg_lambda=0.5,
            subsample=0.7, colsample_bytree=0.5,
            random_state=42, n_jobs=-1),
        'CatBoost': cat.CatBoostRegressor(
            iterations=800, learning_rate=0.05, depth=3,
            l2_leaf_reg=8, subsample=0.7,
            random_seed=42, verbose=0, allow_writing_files=False),
    }
 

Y_Quality는 값의 범위가 극히 좁다 (0.50~0.58, std 매우 작음)
과도한 정규화는 미세한 분산 대비 분할 이득을 짓눌러 트리가 갈라지지 않고 상수만 예측하는 문제 발생
(1) 학습 시 타겟을 표준화해 분할이득을 정규화 대비 증가
(2) 정규화 강도 완화해 최소한의 분할을 가능하게 함

### 클래스 가중치 </br>
- 표본 가중치로 변환
- 데이터 합성 없음

In [4]:
def class_sample_weight(y_class):
    vc = pd.Series(y_class).value_counts()
    w_map = {c: len(y_class)/(len(vc) * n) for c, n in vc.items()}
    return np.array([w_map[c] for c in y_class])

### 임계값 계산</br>
- train fold 내에서만 계산 (누수 차단)

In [5]:
def fit_thresholds(y_quality_tr, y_class_tr):
    q0 = y_quality_tr[y_class_tr == 0]
    q1 = y_quality_tr[y_class_tr == 1]
    q2 = y_quality_tr[y_class_tr == 2]
    thr01 = (q0.max() + q1.min()) / 2 if len(q0) and len(q1) else np.median(y_quality_tr)
    thr12 = (q1.max() + q2.min()) / 2 if len(q1) and len(q2) else np.median(y_quality_tr)
    thr01, thr12 = sorted([thr01, thr12])
    return thr01, thr12

def to_class(pred_quality, thr01, thr12):
    return np.where(pred_quality < thr01, 0, np.where(pred_quality < thr12, 1, 2))

### 라인 x 모델 평가</br>
- Stratified K-Fold
- Fold 내부 MI 선택

In [6]:
def eval_line_model(sub, feats, model_name, model_builder, n_splits):
    X = sub[feats]
    yq = sub['Y_Quality'].values
    yc = sub['Y_Class'].values
    n = len(sub)
    k_feat = 60
    skf_outer = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_pred = np.full(n,-1)
    tr_f1s, te_f1s = [], []
    
    for outer_train_idx, outer_test_idx in skf_outer.split(X,yc):
        skf_inner = StratifiedKFold(n_splits=4, shuffle=True, random_state=1)
        inner_tr_idx, inner_val_idx = next(skf_inner.split(X.iloc[outer_train_idx], yc[outer_train_idx]))
        inner_tr = outer_train_idx[inner_tr_idx]
        inner_val = outer_train_idx[inner_val_idx]
 
        n_train_actual = len(inner_tr)
        k_feat = max(10, min(len(feats), n_train_actual // 5))
                     
        # 피쳐 선택
        X_it_imp = X.iloc[inner_tr].fillna(X.iloc[inner_tr].median()).fillna(0)
        mi = mutual_info_regression(X_it_imp, yq[inner_tr], random_state=42)
        top = X.columns[np.argsort(mi)[::-1][:k_feat]]
        X_it, X_iv, X_te = X.iloc[inner_tr][top], X.iloc[inner_val][top], X.iloc[outer_test_idx][top]

        # 가중치 (class weight)
        w_it = class_sample_weight(yc[inner_tr])

        # 임계값
        thr01, thr12 = fit_thresholds(yq[inner_tr], yc[inner_tr])

        # 타겟 표준화
        mu, sd = yq[inner_tr].mean(), yq[inner_tr].std()
        sd = sd if sd > 1e-8 else 1.0
        yq_it_s = (yq[inner_tr] - mu) / sd

        # 회귀 학습
        model = model_builder()
        if model_name == 'LightGBM':
            model.fit(X_it, yq_it_s, sample_weight=w_it,
                      eval_set=[(X_iv, (yq[inner_val] - mu) / sd)], eval_metric='l2',
                      callbacks=[lgb.early_stopping(50, verbose=False)])
        elif model_name == 'XGBoost':
            model.fit(X_it, yq_it_s, sample_weight=w_it,
                      eval_set=[(X_iv, (yq[inner_val] - mu) / sd)], verbose=False)
        else:  # CatBoost
            model.fit(X_it, yq_it_s, sample_weight=w_it,
                       eval_set=(X_iv, (yq[inner_val] - mu) / sd), early_stopping_rounds=50, verbose=False)
            
        
        tr_pred_c = to_class(model.predict(X_it) * sd + mu, thr01, thr12)
        te_pred_c = to_class(model.predict(X_te) * sd + mu, thr01, thr12)
        oof_pred[outer_test_idx] = te_pred_c
        tr_f1s.append(f1_score(yc[inner_tr], tr_pred_c, average='macro'))
        te_f1s.append(f1_score(yc[outer_test_idx], te_pred_c, average='macro'))

    # 평가 
    macro_f1 = f1_score(yc, oof_pred, average='macro')
    macro_rec = recall_score(yc, oof_pred, average='macro', zero_division=0)
    macro_pre = precision_score(yc, oof_pred, average='macro', zero_division=0)
    gap = np.mean(tr_f1s) - np.mean(te_f1s)

    return {'n': n, 'p_selected': k_feat, 'p/n': round(k_feat / n_train_actual, 2),
            'MacroF1': round(macro_f1, 3), 'Recall': round(macro_rec, 3), 'Precision': round(macro_pre, 3),
            'TrainF1': round(np.mean(tr_f1s), 3), 'Gap': round(gap, 3),
            '과적합': '⚠️' if gap > 0.15 else '✅'}


### 전체 실행

### 종합 성능</br>
- 단순 평균: 라인 하나하나를 동등하게 취급
- 가중 평균: 표본 많은 라인의 안정적 추정에 비중

In [7]:
if __name__ == '__main__':
    lines = load_data()
    models = {'LightGBM' :lambda: build_model()['LightGBM'],
              'XGBoost' :lambda: build_model()['XGBoost'],
              'CatBoost' :lambda: build_model()['CatBoost']}
    
    rows = []
    for ln, sub, feats in lines:
        min_class = pd.Series(sub['Y_Class']).value_counts().min()
        n_splits = 5 if min_class >= 15 else 3
        print(f'라인 {ln} - 샘플 수: {len(sub)}, 원본피처={len(feats)}, 클래스 최소 샘플 수: {min_class}, KFold 분할 수: {n_splits}')
        for mname in ['LightGBM', 'XGBoost', 'CatBoost']:
            res = eval_line_model(sub, feats, mname, models[mname], n_splits)
            res.update({'LINE':ln, 'MODEL':mname})
            rows.append(res)
            print(f'{mname:9s}: MacroF1={res["MacroF1"]:.3f}  Recall={res["Recall"]:.3f}  '
                  f'Precision={res["Precision"]:.3f}  gap={res["Gap"]:+.3f} {res["과적합"]}')
            
    result = pd.DataFrame(rows)[['LINE','MODEL','n','p_selected','p/n','MacroF1','Recall','Precision','TrainF1','Gap','과적합']]

    print('\n라인 x 모델 종합 결과\n')
    print(result.to_string(index=False))
    result.to_csv('baseline_modeling_results.csv', index=False)
        
    print('\n 라인별 최고 모델 (MacroF1 기준)')
    best = result.loc[result.groupby('LINE')['MacroF1'].idxmax()]
    print(best[['LINE','MODEL','MacroF1','Recall','Precision','Gap']].to_string(index=False))

    print('\n\n모델별 종합 성능 (6개 라인 통합)')
    summary_simple = result.groupby('MODEL')[['MacroF1','Recall','Precision','Gap']].mean().round(3)
    summary_simple['과적합_평균'] = summary_simple['Gap'].apply(lambda g: '⚠️' if g > 0.15 else '✅')
 
    def wavg(g, col):
        return np.average(g[col], weights=g['n'])
    summary_w = result.groupby('MODEL').apply(
        lambda g: pd.Series({
            'MacroF1_w': round(wavg(g,'MacroF1'),3),
            'Recall_w': round(wavg(g,'Recall'),3),
            'Precision_w': round(wavg(g,'Precision'),3),
            'Gap_w': round(wavg(g,'Gap'),3),
        })
    ).round(3)
 
    combined = summary_simple[['MacroF1','Recall','Precision','Gap']].join(summary_w)
    combined['과적합'] = combined['Gap_w'].apply(lambda g: '⚠️' if g > 0.15 else '✅')
    combined = combined.rename(columns={'MacroF1':'MacroF1_단순평균','Recall':'Recall_단순평균',
                                         'Precision':'Precision_단순평균','Gap':'Gap_단순평균'})
    print(combined.to_string())
    combined.to_csv('model_summary.csv')
 
    print('\n※ 단순평균 = 6개 라인을 동등 가중 / 가중평균(_w) = 라인 표본수(n)로 가중')
    print('※ 라인별 세부 결과는 line_model_results.csv 참고')

라인 T050304 - 샘플 수: 91, 원본피처=548, 클래스 최소 샘플 수: 11, KFold 분할 수: 3
LightGBM : MacroF1=0.661  Recall=0.670  Precision=0.657  gap=+0.087 ✅
XGBoost  : MacroF1=0.695  Recall=0.681  Precision=0.715  gap=+0.263 ⚠️
CatBoost : MacroF1=0.671  Recall=0.658  Precision=0.688  gap=+0.224 ⚠️
라인 T050307 - 샘플 수: 68, 원본피처=548, 클래스 최소 샘플 수: 12, KFold 분할 수: 3
LightGBM : MacroF1=0.518  Recall=0.496  Precision=0.568  gap=+0.168 ⚠️
XGBoost  : MacroF1=0.683  Recall=0.667  Precision=0.721  gap=+0.343 ⚠️
CatBoost : MacroF1=0.544  Recall=0.548  Precision=0.803  gap=+0.238 ⚠️
라인 T010306 - 샘플 수: 84, 원본피처=548, 클래스 최소 샘플 수: 8, KFold 분할 수: 3
LightGBM : MacroF1=0.447  Recall=0.444  Precision=0.457  gap=+0.182 ⚠️
XGBoost  : MacroF1=0.542  Recall=0.559  Precision=0.534  gap=+0.374 ⚠️
CatBoost : MacroF1=0.531  Recall=0.523  Precision=0.592  gap=+0.329 ⚠️
라인 T010305 - 샘플 수: 73, 원본피처=548, 클래스 최소 샘플 수: 16, KFold 분할 수: 5
LightGBM : MacroF1=0.531  Recall=0.533  Precision=0.592  gap=+0.109 ✅
XGBoost  : MacroF1=0.599  Recall=0.61